# Query quality

Validates the eval dataset against one run: the **wrong queries finder** flags
queries that fail at every $\alpha$ (potentially mislabeled golds), the
**audit view** dumps each failure for manual classification. Works on a single
run — pick it below.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd

from minerva_eval import all_run_dirs, get_run_summary, load_run, best_rank_by_source, EvalResult, Hit

ROOTS = sorted(Path("../experiments").glob("*/runs"))   # every experiment's runs
run_dirs = all_run_dirs(ROOTS)
print(f"{len(run_dirs)} runs from {len(ROOTS)} experiments")

7 runs from 3 experiments


## Run summary

One row per run folder: collection, timestamp, sweep matrix, query count.

In [2]:
run_summary = get_run_summary(ROOTS)
run_summary

,folder,collection,timestamp,top_k,candidate_pool_size,sweep,queries,rows
0,2026-08-27T08-43-19Z_wikipedia-v1,wikipedia-nollm,2026-08-27T08:43:19Z,50,100,"alpha=[0.3, 0.5, 0.7], reranker=[False]",84,252
1,2026-08-27T08-45-52Z_wikipedia-v1,wikipedia-qwen2-5,2026-08-27T08:45:52Z,50,100,"alpha=[0.3, 0.5, 0.7], reranker=[False]",84,252
2,2026-08-30T08-59-15Z_wikipedia-v1,wikipedia-nollm,2026-08-30T08:59:15Z,50,100,"alpha=[0.3, 0.5, 0.7], reranker=[False]",84,252
3,2026-09-02T05-25-15Z_wikipedia-v1,wp-none,2026-09-02T05:25:15Z,50,100,"alpha=[0.3, 0.5, 0.7], reranker=[False]",84,252
4,2026-09-02T05-27-25Z_wikipedia-v1,wp-breadcrumb,2026-09-02T05:27:25Z,50,100,"alpha=[0.3, 0.5, 0.7], reranker=[False]",84,252
5,2026-09-02T05-29-40Z_wikipedia-v1,wp-breadcrumb,2026-09-02T05:29:40Z,50,100,"alpha=[0.3, 0.5, 0.7], reranker=[True]",84,252
6,2026-09-02T05-30-00Z_wikipedia-v1,wp-none,2026-09-02T05:30:00Z,50,100,"alpha=[0.3, 0.5, 0.7], reranker=[True]",84,252


In [4]:
import ipywidgets as widgets

picker = widgets.Dropdown(
    options=[(f"{d.parent.parent.name}/{d.name}", d) for d in run_dirs],
    description="run", layout=widgets.Layout(width="700px"))
display(picker)

Dropdown(description='run', layout=Layout(width='700px'), options=(('baseline/2026-08-27T08-43-19Z_wikipedia-v…

In [5]:
results = load_run(picker.value)
collection = results[0].collection
print(f"{len(results)} rows from {collection!r} ({len({r.query_id for r in results})} queries)")

252 rows from 'wikipedia-nollm' (84 queries)


## Wrong queries finder

Pick failures for R@10 for any $\alpha$ as they may potentially identify wrong gold sources

In [6]:
from collections import defaultdict

def persistent_failures(results, metric: str, threshold: float = 1.0):
    by_query = defaultdict(list)
    for r in results:
        by_query[r.query_id].append(r)
    return [rs for rs in by_query.values()
            if all(getattr(r.metrics, metric) < threshold for r in rs)]

metric = "recall_at_10"
failures = persistent_failures(results, metric)

print(f"{len(failures)} suspect queries")
pd.DataFrame([
    {
     "query_id": rs[0].query_id,
     "query": rs[0].query,
     **{f"α={r.cell.hybrid_alpha}": getattr(r.metrics, metric) for r in sorted(rs, key=lambda r: r.cell.hybrid_alpha)},
    }
    for rs in failures
])

0 suspect queries


""


## Audit view

For each failing query: the gold article(s) with a corpus snippet, plus the
deduplicated top-10 retrieved articles (rank + GOLD marker). Lets each miss be
classified as a real retrieval failure or a mislabeled gold.

The corpus path is read from `config.local.json` (gitignored, machine-specific).
Copy `config.local.json.example` and set `corpus_dir`.

In [7]:
config = json.loads(Path("config.local.json").read_text())
CORPUS_DIR = Path(config["corpus_dir"])

def read_snippet(source_id: str, max_chars: int = 300) -> str:
    text = (CORPUS_DIR / source_id).read_text()
    body = re.sub(r"^---\n.*?\n---\n", "", text, count=1, flags=re.DOTALL)  # drop frontmatter
    paras = [s for p in body.split("\n\n") if (s := p.strip()) and not s.startswith("#")]
    snippet = paras[0] if paras else ""
    return snippet[:max_chars] + ("…" if len(snippet) > max_chars else "")

def top_articles(hits: list[Hit], k: int = 10) -> list[Hit]:
    seen, out = set(), []
    for h in sorted(hits, key=lambda h: h.rank):
        if h.source_id not in seen:
            seen.add(h.source_id)
            out.append(h)
        if len(out) == k:
            break
    return out

def audit(groups: list[list[EvalResult]]) -> None:
    for rs in groups:
        rs = sorted(rs, key=lambda r: r.cell.hybrid_alpha)
        rep = rs[1]  # representative for the top-10 dump
        print(f"=== {rep.query_id} ===")
        print(f"query: {rep.query!r}")
        for g in rep.gold_sources:
            ranks = {r.cell.hybrid_alpha: best_rank_by_source(r).get(g, "—") for r in rs}
            print(f"gold  [{g}] ranks {ranks}: {read_snippet(g)}")
        print(f"top-10 articles (α={rep.cell.hybrid_alpha}):")
        for h in top_articles(rep.hits):
            mark = "GOLD" if h.source_id in rep.gold_sources else "    "
            print(f"  {h.rank:>2} {mark} {h.source_id}  ({h.score:.4f})")
        print()

audit(failures)